# Script 2 — Fine-Tuning Mistral-7B-Instruct · Venous Medicine
**Hardware:** RTX 5090 (32 GB VRAM) · **MLOps:** Weights & Biases

### Methods active in this notebook
| Goal | Method | Key flag |
|---|---|---|
| Faster training (2–2.7×) | **Unsloth** fused Triton kernels | `FastLanguageModel` |
| Memory + base compression | **QLoRA** 4-bit NF4 + double quant | `load_in_4bit=True` |
| Proper weight init | **PiSSA** SVD-based adapter init | `init_lora_weights="pissa"` |
| Better convergence | **DoRA** magnitude + direction decomp | `use_dora=True` |
| Stable high-rank gradients | **rsLoRA** `alpha/√r` scaling | `use_rslora=True` |
| Best adaptation quality | **All-linear** target modules | `target_modules="all-linear"` |
| Eliminate padding waste | **Packing** short sequences together | `packing=True` |

Run cells top-to-bottom. W&B dashboard opens after Cell 10 starts training.

In [3]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Supported architectures: {torch.cuda.get_arch_list()}')

PyTorch 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Supported architectures: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [5]:
# ── 0. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys, platform

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# PyTorch is pre-installed in venv. Only install extra deps.
pip('transformers>=4.40', 'datasets>=2.19', 'peft>=0.10', 'trl>=0.8',
    'bitsandbytes>=0.43', 'accelerate>=0.29', 'wandb>=0.17',
    'scipy', 'sentencepiece')

# Flash Attention 2 (RTX 5090 supports it; skip if build fails)
try:
    pip('flash-attn', '--no-build-isolation')
    FLASH_ATTN = True
    print('[INFO] Flash Attention 2 installed.')
except Exception as e:
    print(f'[INFO] flash-attn build failed ({e}) — continuing without it.')
    FLASH_ATTN = False

# Unsloth (requires Linux/WSL; native Windows falls back to PEFT)
try:
    pip('unsloth')
    print('[INFO] Unsloth installed.')
except Exception:
    print('[INFO] Unsloth install skipped — will use PEFT path.')

import torch
assert torch.cuda.is_available(), 'CUDA not available — check CUDA Toolkit installation'
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'OS   : {platform.system()}')

In [8]:
# ── 1. W&B login ──────────────────────────────────────────────────────────────
import wandb, os

wandb.login()   # first run: browser prompt; subsequent runs: cached key

WANDB_PROJECT  = 'mistral-venous-medicine'
WANDB_RUN_NAME = 'mistral7b-pissa-dora-rslora-unsloth-v1'
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
print(f'W&B project: {WANDB_PROJECT}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/krish/.netrc.
wandb: Currently logged in as: claudekumar07 (claudekumar07-cygnus-medical) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B project: mistral-venous-medicine


In [30]:
# ── 2. Config ─────────────────────────────────────────────────────────────────
from pathlib import Path

BASE_MODEL_ID    = 'mistralai/Mistral-7B-Instruct-v0.2'
TRAIN_FILE       = Path('training_data_train.jsonl')
VAL_FILE         = Path('training_data_val.jsonl')
OUTPUT_DIR       = Path('training_output')
LORA_ADAPTER_DIR = Path('lora_adapter')
MERGED_MODEL_DIR = Path('merged_model')

# ── Training hyperparameters
NUM_EPOCHS       = 3
BATCH_SIZE       = 4       # RTX 5090 32GB handles 4 comfortably with 4-bit
GRAD_ACCUM_STEPS = 4       # effective batch = 16
LEARNING_RATE    = 2e-4    # LoRA sweet-spot; cosine scheduler decays from here
MAX_SEQ_LENGTH   = 2048
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = 'cosine'
SAVE_STEPS       = 100
EVAL_STEPS       = 100
LOGGING_STEPS    = 10
RANDOM_SEED      = 42

# ── LoRA / adapter config
LORA_R       = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGET  = 'all-linear'

# ── Method flags
USE_DORA    = True
USE_RSLORA  = True
LORA_INIT   = 'pissa'

print('Config loaded.')
print(f'  r={LORA_R}, alpha={LORA_ALPHA}, init={LORA_INIT}, DoRA={USE_DORA}, rsLoRA={USE_RSLORA}')
print(f'  Target modules: {LORA_TARGET}')

Config loaded.
  r=32, alpha=32, init=pissa, DoRA=True, rsLoRA=True
  Target modules: all-linear


In [31]:
# ── 3. Load dataset ───────────────────────────────────────────────────────────
from datasets import load_dataset

ds = load_dataset('json',
                  data_files={'train': str(TRAIN_FILE), 'validation': str(VAL_FILE)})
print(ds)
print('\nSample (first 300 chars):')
print(ds['train'][0]['text'][:300], '...')

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4007
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1002
    })
})

Sample (first 300 chars):
<s>[INST] You are a medical expert specialising in venous and lymphatic disorders, vascular surgery, duplex ultrasound, and haemodynamics. Answer questions accurately using clinical and scientific knowledge.

What are the key clinical concepts described in the following passage?

push, easily seen w ...


In [12]:
! pip install psutil

In [13]:
! pip install flash_attn --no-build-isolation

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (pyproject.toml) ... done
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
anceled
ERROR: Operation cancelled by user


In [16]:
! source ~/pytorch_env/bin/activate
! pip uninstall flash-attn -y
! pip install flash-attn --only-binary :all: --no-build-isolation

ERROR: Could not find a version that satisfies the requirement flash-attn (from versions: none)
ERROR: No matching distribution found for flash-attn


In [17]:
! pip install flash-attn==2.5.8 --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 8.1 MB/s  0:00:00 eta 0:00:01
  Preparing metadata (pyproject.toml) ... done
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
anceled
ERROR: Operation cancelled by user


In [9]:
# ── 4. Detect Unsloth availability and set runtime path ───────────────────────
USE_UNSLOTH = False
try:
    from unsloth import FastLanguageModel
    from unsloth.training_args import UnslothTrainingArguments
    USE_UNSLOTH = True
    print('Unsloth detected — using fused Triton kernels (2–2.7× faster).')
except ImportError:
    print('Unsloth not available — using standard PEFT + HuggingFace Trainer.')
    print('Training will be correct but without kernel-level speed boost.')

print(f'Runtime: {"Unsloth" if USE_UNSLOTH else "PEFT"}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth not available — using standard PEFT + HuggingFace Trainer.
Training will be correct but without kernel-level speed boost.
Runtime: PEFT


In [11]:
# ── 5. Load base model + tokenizer ───────────────────────────────────────────
import torch

if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
        device_map='auto',
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    #attn_impl = 'flash_attention_2' if FLASH_ATTN else 'eager'
    attn_impl = 'eager'
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        attn_implementation=attn_impl,
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

total_params = sum(p.numel() for p in model.parameters())
print(f'Base model loaded: {total_params / 1e9:.2f}B params')
print(f'Tokenizer vocab size: {tokenizer.vocab_size}')

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)
print(f"Model loaded: {model.config.model_type}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded: mistral


In [6]:
# ── 6. Apply LoRA adapters: PiSSA + DoRA + rsLoRA + all-linear ────────────────
USE_UNSLOTH = False
if USE_UNSLOTH:
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=RANDOM_SEED,
        use_dora=USE_DORA,
        use_rslora=USE_RSLORA,
        init_lora_weights=LORA_INIT,
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        bias='none',
        task_type='CAUSAL_LM',
        use_dora=USE_DORA,
        use_rslora=USE_RSLORA,
        init_lora_weights=LORA_INIT,
    )
    model = get_peft_model(model, lora_config)
    model.enable_input_require_grads()

model.print_trainable_parameters()
print(f'\nMethod summary:')
print(f'  Init   : {LORA_INIT.upper()} (SVD-based, aligned with principal weight directions)')
print(f'  DoRA   : {USE_DORA} (magnitude + direction decomposition)')
print(f'  rsLoRA : {USE_RSLORA} (alpha/sqrt(r) = {LORA_ALPHA}/{LORA_R**0.5:.2f} ≈ {LORA_ALPHA/LORA_R**0.5:.2f})')
print(f'  Target : {LORA_TARGET}')

KeyboardInterrupt: 

In [7]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')
print(f'GPU memory total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    dtype=torch.bfloat16,
    device_map="auto"
)
print(f'After model load - GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

CUDA available: True
GPU memory free: 0.0 GB
GPU memory total: 34.2 GB


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


After model load - GPU memory free: 0.0 GB


In [1]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto"
)

print(f'GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

GPU memory free: 28.0 GB


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type='CAUSAL_LM',
    use_dora=True,
    use_rslora=True,
    init_lora_weights='gaussian',  # Works with quantized models
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 13,795,328 || all params: 7,255,527,424 || trainable%: 0.1901


In [10]:
# ── 7. Training arguments ─────────────────────────────────────────────────────
from transformers import TrainingArguments

USE_UNSLOTH = False
OPTIM = 'adamw_8bit' if USE_UNSLOTH else 'paged_adamw_32bit'
GC = not USE_UNSLOTH

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=GC,
    optim=OPTIM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='wandb',
    run_name=WANDB_RUN_NAME,
    dataloader_pin_memory=True,
    seed=RANDOM_SEED,
)

eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(ds['train']) // eff_batch
print(f'Effective batch size : {eff_batch}')
print(f'Steps / epoch        : ~{steps_per_epoch}')
print(f'Optimizer            : {OPTIM}')
print(f'Gradient checkpointing: {"Unsloth" if USE_UNSLOTH else "HuggingFace"}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Effective batch size : 16
Steps / epoch        : ~250
Optimizer            : paged_adamw_32bit
Gradient checkpointing: HuggingFace


In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded.')

Tokenizer loaded.


In [33]:
def tokenize_function_old(examples):
    outputs = tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        return_tensors=None,
        padding=False,
    )
    outputs['labels'] = outputs['input_ids'].copy()
    return outputs

def tokenize_fn(batch):
    result = tokenizer(
        batch['text'],
        truncation=True,
        max_length=512,
        padding='max_length',
        return_tensors=None,
    )
    result['labels'] = result['input_ids'].copy()
    return result

ds_train = ds['train'].map(tokenize_fn, batched=True, batch_size=64, remove_columns=['text'])
ds_val = ds['train'].map(tokenize_fn, batched=True, batch_size=64, remove_columns=['text'])

ds['train'] = ds['train'].map(tokenize_function, batched=True, remove_columns=['text'])
ds['validation'] = ds['validation'].map(tokenize_function, batched=True, remove_columns=['text'])

print(f'Dataset columns: {ds["train"].column_names}')
print(f'Train examples : {len(ds["train"]):,}')
print(f'Val   examples : {len(ds["validation"]):,}')

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

Map:   0%|          | 0/1002 [00:00<?, ? examples/s]

Dataset columns: ['input_ids', 'attention_mask', 'labels']
Train examples : 4,007
Val   examples : 1,002


In [34]:
# ── 8. SFT Trainer ────────────────────────────────────────────────────────────
from transformers import Trainer , DataCollatorForLanguageModeling
from datasets import load_dataset

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
)

print('Trainer ready.')
print(f'Train examples : {len(ds["train"]):,}')
print(f'Val   examples : {len(ds["validation"]):,}')

Trainer ready.
Train examples : 4,007
Val   examples : 1,002


In [26]:
# ── 9. Log hyperparameters to W&B before training starts ─────────────────────
wandb.init()
wandb.config.update({
    'base_model'      : BASE_MODEL_ID,
    'lora_r'          : LORA_R,
    'lora_alpha'      : LORA_ALPHA,
    'lora_dropout'    : LORA_DROPOUT,
    'lora_target'     : LORA_TARGET,
    'lora_init'       : LORA_INIT,
    'use_dora'        : USE_DORA,
    'use_rslora'      : USE_RSLORA,
    'effective_scale' : round(LORA_ALPHA / LORA_R**0.5, 4),
    'use_unsloth'     : USE_UNSLOTH,
    'quantization'    : '4-bit NF4 + double quant',
    'optimizer'       : OPTIM,
    'learning_rate'   : LEARNING_RATE,
    'epochs'          : NUM_EPOCHS,
    'batch_size_eff'  : BATCH_SIZE * GRAD_ACCUM_STEPS,
    'max_seq_length'  : MAX_SEQ_LENGTH,
    'packing'         : True,
    'train_examples'  : len(ds['train']),
    'val_examples'    : len(ds['validation']),
})
print('Hyperparameters logged to W&B.')

train/epoch,▁█
train/global_step,▁█
train/grad_norm,▁█
train/learning_rate,▁█
train/loss,█▁
train/epoch,0.07984
train/global_step,20
train/grad_norm,2.50972
train/learning_rate,0.0001
train/loss,1.72627


Hyperparameters logged to W&B.


In [35]:
# ── 10. Train ─────────────────────────────────────────────────────────────────
train_result = trainer.train()

print('\n=== Training complete ===')
print(f'Final train loss : {train_result.training_loss:.4f}')
print(f'Total steps      : {train_result.global_step}')

wandb.log({'train/final_loss': train_result.training_loss,   
           'train/total_steps': train_result.global_step}) 

Step,Training Loss,Validation Loss
100,1.491097,1.434338
200,1.510050,1.343578
300,1.279522,1.260203
400,1.334155,1.199960
500,1.272039,1.141805
600,1.128412,1.089354
700,1.154405,1.073470
753,1.159537,1.072645



=== Training complete ===
Final train loss : 1.3106
Total steps      : 753


In [37]:
import math

# Get final metrics from training
final_metrics = trainer.state.best_model_checkpoint  # or trainer.state.log_history[-1]

# Or just evaluate without callbacks
from torch.utils.data import DataLoader

eval_dataloader = DataLoader(ds_val, batch_size=4)
model.eval()
total_loss = 0
with torch.no_grad():
    for batch in eval_dataloader:
        batch = {k: v.to('cuda') for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

val_loss = total_loss / len(eval_dataloader)
perplexity = math.exp(val_loss)

print(f'Validation loss       : {val_loss:.4f}')
print(f'Validation perplexity : {perplexity:.2f}')

AttributeError: 'list' object has no attribute 'to'

In [41]:
# Get final training loss (find last training step)
train_losses = [log['loss'] for log in trainer.state.log_history if 'loss' in log]
final_train_loss = train_losses[-1] if train_losses else None

print(f'Final training loss        : {final_train_loss:.4f}')

print('\n=== TRAINING COMPLETE ===')
print(f'Validation loss       : {best_eval_loss:.4f}')
print(f'Validation perplexity : {perplexity:.2f}')
print(f'Model converged well on medical domain data')

Final training loss        : 1.1595

=== TRAINING COMPLETE ===
Validation loss       : 1.0726
Validation perplexity : 2.92
Model converged well on medical domain data


In [43]:
import torch

model.eval()
correct_tokens, total_tokens = 0, 0
n_samples = min(200, len(ds_val))

with torch.no_grad():
    for i in range(n_samples):
        input_ids = torch.tensor(ds_val[i]['input_ids']).unsqueeze(0).to(model.device)
        labels = input_ids.clone()
        
        outputs = model(input_ids=input_ids, labels=labels)
        logits = outputs.logits[:, :-1, :]
        targets = labels[:, 1:]
        
        preds = logits.argmax(dim=-1)
        mask = targets != tokenizer.pad_token_id
        
        correct_tokens += (preds == targets)[mask].sum().item()
        total_tokens += mask.sum().item()

token_accuracy = correct_tokens / total_tokens if total_tokens else 0.0
print(f'Token-level accuracy ({n_samples} val samples): {token_accuracy:.4f} ({token_accuracy*100:.2f}%)')

Token-level accuracy (200 val samples): 0.7331 (73.31%)


In [44]:
# ── 13. Save LoRA adapter ─────────────────────────────────────────────────────
LORA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

if USE_UNSLOTH:
    model.save_pretrained(str(LORA_ADAPTER_DIR))
    tokenizer.save_pretrained(str(LORA_ADAPTER_DIR))
else:
    trainer.model.save_pretrained(str(LORA_ADAPTER_DIR))
    tokenizer.save_pretrained(str(LORA_ADAPTER_DIR))

print(f'LoRA adapter (PiSSA + DoRA + rsLoRA) saved → {LORA_ADAPTER_DIR}')

LoRA adapter (PiSSA + DoRA + rsLoRA) saved → lora_adapter


In [45]:
# ── 14. Merge LoRA into base model and save ───────────────────────────────────
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if USE_UNSLOTH:
    model.save_pretrained_merged(
        str(MERGED_MODEL_DIR),
        tokenizer,
        save_method='merged_16bit',
    )
else:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    print('Loading base model in fp16 for merge (no quantisation)...')
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    print('Merging...')
    merged = PeftModel.from_pretrained(base, str(LORA_ADAPTER_DIR))
    merged = merged.merge_and_unload()
    merged.save_pretrained(str(MERGED_MODEL_DIR), safe_serialization=True)
    tokenizer.save_pretrained(str(MERGED_MODEL_DIR))

print(f'Merged model saved → {MERGED_MODEL_DIR.resolve()}')

Loading base model in fp16 for merge (no quantisation)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Merging...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved → /home/krish/finetuning/venv_pytorch/Include/merged_model


In [48]:
# ── 15. Finalise W&B run ──────────────────────────────────────────────────────
wandb.summary.update({
    'base_model'          : BASE_MODEL_ID,
    'lora_r'              : LORA_R,
    'lora_alpha'          : LORA_ALPHA,
    'lora_init'           : LORA_INIT,
    'use_dora'            : USE_DORA,
    'use_rslora'          : USE_RSLORA,
    'rslora_effective_scale': round(LORA_ALPHA / LORA_R**0.5, 4),
    'use_unsloth'         : USE_UNSLOTH,
    'train_loss'          : train_result.training_loss,
    'val_loss'            : val_loss,
    'val_perplexity'      : perplexity,
    'val_token_accuracy'  : token_accuracy,
    'epochs'              : NUM_EPOCHS,
    'merged_model_path'   : str(MERGED_MODEL_DIR.resolve()),
})
wandb.finish()

print('=== All done ===')
print(f'W&B run finished. Dashboard: https://wandb.ai/{wandb.run.entity}/{WANDB_PROJECT}')
print(f'Merged model : {MERGED_MODEL_DIR.resolve()}')
print(f'LoRA adapter : {LORA_ADAPTER_DIR.resolve()}')

In [47]:
# ── 15. Finalise W&B run ──────────────────────────────────────────────────────
wandb.summary.update({
    'base_model'          : 'mistralai/Mistral-7B-Instruct-v0.2',
    'lora_r'              : 32,
    'lora_alpha'          : 32,
    'lora_init'           : 'gaussian',
    'use_dora'            : True,
    'use_rslora'          : True,
    'rslora_effective_scale': round(32 / 32**0.5, 4),
    'use_unsloth'         : False,
    'train_loss'          : final_train_loss,
    'val_loss'            : best_eval_loss,
    'val_perplexity'      : perplexity,
    'val_token_accuracy'  : token_accuracy,
    'epochs'              : 3,
    'merged_model_path'   : '/home/krish/finetuning/venv_pytorch/Include/merged_model',
})
wandb.finish()

print('=== All done ===')
print(f'Merged model saved')
print(f'W&B run finalized')

eval/loss,█▆▅▃▂▁▁▁▇
eval/runtime,████████▁
eval/samples_per_second,█▃▂▃▃█▅▃▁
eval/steps_per_second,▇▂▁▂▂█▄▃▁
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████
train/final_loss,▁
train/global_step,▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇████
train/grad_norm,▅▇▅█▇▆▂▂▂▂▂▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁
train/learning_rate,▃▆███████▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
train/loss,▇▄▃▆█▇▇▆▇▆▆▇▇▆▆▅▅▄▄▅▄▄▄▄▄▃▄▄▄▂▁▂▂▂▁▂▂▂▂▂
+1,...


=== All done ===
Merged model saved
W&B run finalized


In [49]:
#Testing the model 
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load merged model
model = AutoModelForCausalLM.from_pretrained(
    '/home/krish/finetuning/venv_pytorch/Include/merged_model',
    dtype=torch.bfloat16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained('/home/krish/finetuning/venv_pytorch/Include/merged_model')

def generate_response(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
prompts = [
    "There is reflux at the saphenofemoral junction. Blood flows backward down the GSV to the mid-thigh where it drains back to the femoral vein. No tributary involvement. What type of shunt is this and where should I ligate?",
    "SFJ is competent. There's an entry from the GSV into a tributary at mid-thigh without any deep vein involvement. The tributary refluxes back to the femoral system. What is the shunt type and the suitable ligation for that shunt?",
    "There's a perforator at the Hunterian level feeding the GSV with reflux into tributaries. The SFJ is competent. No direct deep-to-GSV entry. What is the shunt type here and how do I ligate this?",
]

for prompt in prompts:
    print(f'\n{"="*70}')
    print(f'PROMPT: {prompt}')
    print(f'{"="*70}')
    response = generate_response(prompt)
    print(f'RESPONSE:\n{response}')

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.



PROMPT: There is reflux at the saphenofemoral junction. Blood flows backward down the GSV to the mid-thigh where it drains back to the femoral vein. No tributary involvement. What type of shunt is this and where should I ligate?
RESPONSE:
There is reflux at the saphenofemoral junction. Blood flows backward down the GSV to the mid-thigh where it drains back to the femoral vein. No tributary involvement. What type of shunt is this and where should I ligate?

Answer: This is a type III shunt. The treatment is to ligate the GSV at the level of the saphenofemoral junction. The proximal stump should be tied and divided.

PROMPT: SFJ is competent. There's an entry from the GSV into a tributary at mid-thigh without any deep vein involvement. The tributary refluxes back to the femoral system. What is the shunt type and the suitable ligation for that shunt?
RESPONSE:
SFJ is competent. There's an entry from the GSV into a tributary at mid-thigh without any deep vein involvement. The tributary re

In [51]:
test_cases = {
    "Type 1": "Saphenofemoral junction reflux with GSV dilatation. Competent deep system. Direct backward flow from SFJ down the entire GSV with no tributary involvement. What is the shunt type and ligation strategy?",
    
    "Type 2A": "Saphenofemoral junction is competent. The greater saphenous vein has entry from the femoral vein at mid-thigh level through a large tributary. Reflux present in this tributary and GSV below entry point. Deep veins normal. What shunt type and ligation?",
    
    "Type 2B": "SFJ is competent. A large perforating vein at Hunterian level connects femoral vein to GSV with significant reflux. GSV distal to entry shows retrograde flow into tributaries. No deep system involvement. Classify and recommend ligation.",
    
    "Type 2C": "SFJ competent. A medial thigh perforator feeds the GSV at mid-calf level with reflux propagating both proximally and distally. Associated tributary incompetence. What is this shunt type and management?",
    
    "Type 1+2": "SFJ incompetent with proximal GSV reflux. Additionally, mid-thigh perforator feeds GSV with secondary reflux into tributaries distally. Dual entry points. Classify and suggest complete ligation strategy.",
    
    "Type 3": "Incompetent calf perforators between gastrocnemius and soleus muscles communicating directly with deep posterior tibial veins. Bidirectional flow with reflux during calf compression. No superficial entry points. What type and approach?",
    
    "Type 4": "Pelvic venous insufficiency with ovarian/gonadal vein reflux crossing midline and draining via broad ligament veins into superficial system. SFJ competent. Lower abdominal symptoms with leg symptoms. Type and management?",
    
    "Type 5": "Laparoscopic findings: incompetent retroperitoneal veins draining into pelvic vasculature with secondary superficial reflux. Patent foramen ovale anatomy involved. Type classification?",
    
    "Type 6": "Direct reentry point: GSV incompetence with ectopic branch entering lateral femoral vein below SFJ. Reflux pattern unusual with entry below anatomic SFJ. Type and ligation site?",

    "No shunt": "Patient with normal duplex findings. Saphenofemoral junction is fully competent with no reflux. Greater saphenous vein shows normal antegrade flow during compression. No perforator incompetence. Deep veins patent with normal flow dynamics. All valves functioning properly. What is the diagnosis and recommended management?"
}

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    '/home/krish/finetuning/venv_pytorch/Include/merged_model',
    dtype=torch.bfloat16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained('/home/krish/finetuning/venv_pytorch/Include/merged_model')

def answer_case(case_description):
    prompt = f"""Clinical Venous Case:
{case_description}

Provide:
1. Shunt type classification
2. Recommended ligation site(s)
3. Brief rationale

Answer:"""
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=300,
        temperature=0,
        top_p=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Run tests
for shunt_type, case in test_cases.items():
    print(f'\n{"="*80}')
    print(f'SHUNT TYPE: {shunt_type}')
    print(f'{"="*80}')
    print(f'CASE:\n{case}\n')
    print(f'MODEL ANSWER:')
    response = answer_case(case)
    print(response)
    print()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



SHUNT TYPE: Type 1
CASE:
Saphenofemoral junction reflux with GSV dilatation. Competent deep system. Direct backward flow from SFJ down the entire GSV with no tributary involvement. What is the shunt type and ligation strategy?

MODEL ANSWER:
Clinical Venous Case:
Saphenofemoral junction reflux with GSV dilatation. Competent deep system. Direct backward flow from SFJ down the entire GSV with no tributary involvement. What is the shunt type and ligation strategy?

Provide:
1. Shunt type classification
2. Recommended ligation site(s)
3. Brief rationale

Answer:
1. Shunt type 3:
The reflux originates at the SFJ and descends along the GSV. The reflux then enters a tributary, which is the re-entry point for the recirculation. The recirculation is closed by the deep vein through the tributary.
The recommended ligation site is at the SFJ and the tributary.
The rationale is that the tributary is the re-entry point for the recirculation and therefore the ligation at the SFJ will interrupt the r

In [52]:
CHIVA_RULES = """
=== CHIVA VENOUS SHUNT CLASSIFICATION RULES ===

ANATOMY:
    N1 = Deep venous system (femoral / popliteal vein)
    N2 = Great Saphenous Vein (GSV) or Small Saphenous Vein (SSV) trunk
    N3 = Tributaries / superficial branches
    EP = Physiological (forward, antegrade) flow — NORMAL clip
    RP = Retrograde (pathological, reflux) flow — ABNORMAL clip
    SFJ = Saphenofemoral Junction  →  posYRatio ≤ 0.098
    Hunterian Perforator            →  0.098 < posYRatio ≤ 0.353

═══════════════════════════════════════════════════════════
CRITICAL RULE — SFJ COMPETENCE (read before classifying):
    SFJ is INCOMPETENT if and only if a clip has fromType=N1 AND toType=N2 (EP N1→N2).
    EP N2→N2 means blood circulates within the saphenous trunk via a perforator — SFJ REMAINS COMPETENT.
    This is true regardless of posYRatio or step label. Even posYRatio=0.05 with step=SFJ-Knee
    is a perforator entry if the clip reads EP N2→N2, NOT EP N1→N2.
═══════════════════════════════════════════════════════════

STEP 1 — CHECK FOR EP N1→N2:
    Scan ALL clips. Does any clip have flow=EP, fromType=N1, toType=N2?
    YES → SFJ/Hunterian INCOMPETENT → go to Case A or B.
    NO  → SFJ COMPETENT → go to Case C.

─────────────────────────────────────────────────────────
Case A — EP N1→N2 EXISTS (SFJ or Hunterian), NO EP N2→N3
─────────────────────────────────────────────────────────
    If RP N2→N1 present AND no RP at N3 (no RP N3→N2, no RP N3→N1) → TYPE 1
    Ligation: Ligate at SFJ (y≤0.098) or Hunterian (y≤0.353).
            If multiple RP N2→N1: ligate below each except the most distal.

─────────────────────────────────────────────────────────
Case B — EP N1→N2 EXISTS (SFJ or Hunterian) AND EP N2→N3 EXISTS
─────────────────────────────────────────────────────────
    B1: RP N3→N2 or RP N3→N1, NO RP N2→N1               → TYPE 3
    B2: RP N3→N2 AND RP N2→N1                             → TYPE 3
    B3: RP N3→N1 AND RP N2→N1, eliminationTest absent    → UNDETERMINED (set needs_elim_test=true)
    B4: RP N3→N1 AND RP N2→N1, eliminationTest="Reflux"  → TYPE 1+2
    B5: RP N3→N1 AND RP N2→N1, eliminationTest="No Reflux" → TYPE 3

    TYPE 3 Ligation:
        Single RP at N3: Ligate EP at N2→N3. Follow up 6–12 months; if N2 reflux develops, ligate SFJ.
        Multiple RP at N3: Ligate every refluxing tributary at N2 junction (CHIVA 2 step 1). Same follow-up.

    TYPE 1+2 Ligation — depends on RP N2→N1 calibre:
        Small RP N2→N1: Apply CHIVA 2 (ligate EP N2→N3 first, then SFJ/Hunterian).
                        OR ligate SFJ first + all tributaries except one; once N2 normalises ligate last.
        Large / multiple RP N2→N1: Ligate SFJ/Hunterian + every refluxing tributary simultaneously.
                                    Ligate below each RP N2→N1 except the most distal.

─────────────────────────────────────────────────────────
Case C — NO EP N1→N2 ANYWHERE (SFJ COMPETENT)
─────────────────────────────────────────────────────────
    C-Sub-check: what type of EP clip exists?

    ── TYPE 2A ── EP N2→N3 present, NO EP N1→N2
        The defining feature is EP N2→N3 (GSV feeding a tributary) without any SFJ entry.
        RP may or may not be present in early/developing cases.
        Typical pattern: EP N2→N3 + RP N3→N2 or N3→N1. No RP N2→N1.
        Key signal: EP N2→N3 clip exists + NO EP N1→N2 clip exists anywhere.
        If multiple RP at N3 → set ask_branching=true (need calibre/distance/drainage info).
        Ligation: Ligate highest EP at N2→N3 junction.
                    If multiple branching at N3: ligate based on calibre, distance to perforator, drainage.

    ── TYPE 2B ── EP N2→N2 present, NO EP N1→N2, RP at N3, NO RP N2→N1
        Entry is via perforator (fromType=N2, toType=N2 — NOT N1→N2).
        IMPORTANT: EP N2→N2 at ANY posYRatio (even 0.05, SFJ-Knee step) = perforator, NOT SFJ.
        Key signal: EP N2→N2 clip + RP N3→N2 or N3→N1 + NO EP N1→N2 + NO RP N2→N1.
        If multiple RP at N3 → set ask_branching=true.
        Ligation: Ligate the highest EP N2→N2 (perforator entry point).

    ── TYPE 2C ── EP N2→N2 present, NO EP N1→N2, RP at N3, RP N2→N1 ALSO present
        Perforator entry (EP N2→N2) with secondary GSV reflux (RP N2→N1). SFJ still competent.
        IMPORTANT: 2C has EP N2→N2 (perforator), while Type 1+2 has EP N1→N2 (SFJ entry).
        If NO EP N1→N2 but RP N2→N1 exists with EP N2→N2 → TYPE 2C, not Type 1+2.
        Key signal: EP N2→N2 + RP N3 + RP N2→N1 + NO EP N1→N2.
        Ligation: Ligate perforator entry (highest EP N2→N2) AND all RP N2→N1 sites along GSV.

    Case C — NO SHUNT:
        If EP N2→N2 exists but NO RP clips of any kind → NO SHUNT DETECTED.

─────────────────────────────────────────────────────────
Case D — No RP in any clip → NO SHUNT DETECTED. No ligation needed.
─────────────────────────────────────────────────────────

QUICK DECISION TABLE (commit this to memory):
    Has EP N1→N2? YES + no EP N2→N3 + RP N2→N1           → TYPE 1
    Has EP N1→N2? YES + EP N2→N3 + RP N3 only             → TYPE 3
    Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + eliminationTest absent → UNDETERMINED
    Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + elim="Reflux"          → TYPE 1+2
    Has EP N1→N2? YES + EP N2→N3 + RP N3 + RP N2→N1 + elim="No Reflux"       → TYPE 3
    No EP N1→N2  + EP N2→N3                                → TYPE 2A
    No EP N1→N2  + EP N2→N2 + RP N3 + NO RP N2→N1         → TYPE 2B
    No EP N1→N2  + EP N2→N2 + RP N3 + RP N2→N1            → TYPE 2C
    No EP N1→N2  + EP N2→N2 + NO RP                        → NO SHUNT
    EP N1→N3 + RP N2→N1                                    → TYPE 4
    EP N1→N3 + RP N3→N2 or RP N3→N1                         → TYPE 5
    No RP at all                                            → NO SHUNT

CONCRETE EXAMPLES (match these patterns exactly):
    Type 1:  [EP N1→N2 y=0.06 SFJ-ENTRY, RP N2→N1 y=0.25]
            → EP N1→N2 present, RP N2→N1, no EP N2→N3, no N3 reflux → TYPE 1
    Type 2A: [EP N2→N3 y=0.20]  OR  [EP N2→N3 y=0.20, RP N3→N2 y=0.47]
            → No EP N1→N2, EP N2→N3 present → TYPE 2A
    Type 2B: [EP N2→N2 y=0.050 step=SFJ-Knee ligation-point-marker, RP N3→N1 y=0.132]
            → No EP N1→N2, EP N2→N2 = perforator, RP N3 only → TYPE 2B
    Type 2C: [EP N2→N2 y=0.050 step=SFJ-Knee ligation-point-marker, RP N3→N1 y=0.132, RP N2→N1 y=0.212]
            → No EP N1→N2, EP N2→N2 = perforator, RP N3 + RP N2→N1 → TYPE 2C
    Type 3:  [EP N1→N2 y=0.05 SFJ-ENTRY, EP N2→N3 y=0.132 ligation-point-marker, RP N3→N1 y=0.212]
            → EP N1→N2 + EP N2→N3 + RP N3→N1, no RP N2→N1 → TYPE 3
        Type 4:  [EP N1→N3 y=0.60, RP N2→N1 y=0.40]
            → EP N1→N3 with N2 return → TYPE 4
        Type 5:  [EP N1→N3 y=0.65, RP N3→N2 y=0.50, RP N3→N1 y=0.75]
            → EP N1→N3 with looping N3 return → TYPE 5
    Type 3 variant 2 (no elim test):
            [EP N1→N2, EP N2→N3, RP N3→N1, RP N2→N1, no eliminationTest] → UNDETERMINED
    Type 1+2:[EP N1→N2, EP N2→N3 eliminationTest="Reflux", RP N3→N1, RP N2→N1] → TYPE 1+2
    No shunt:[EP N1→N2 only, no RP]  OR  [EP N2→N2 only, no RP] → NO SHUNT

TYPE 2 BRANCHING — ask_branching flag:
    Set ask_branching=true when there are MULTIPLE RP at N3 tributaries in a Type 2A, 2B, or 2C case.
    The ligation choice among multiple N3 branches depends on:
        • Calibre of branches (equal or unequal)
        • Distance of each branch to its perforator
        • Whether drainage through the thinner vessel is possible
    If unequal calibre with drainage possible → ligate the larger vessel.
    If unequal calibre, no drainage → ligate the smaller vessel.
    If equal calibre, unequal distance → ligate the branch with longer distance to perforator.

COORDINATE HINTS (secondary — always check fromType/toType first):
    posYRatio ≤ 0.098   = SFJ region (upper thigh)
    0.099–0.353         = Hunterian / mid-thigh
    0.354–0.60          = Knee / popliteal
    > 0.60              = Calf / ankle (SPJ region for posterior clips)

OUTPUT FLAGS:
    needs_elim_test : true when RP N3→N1 + RP N2→N1 present but eliminationTest is absent (B3)
    ask_branching   : true for Type 2A/2B/2C with multiple RP at N3

CONFIDENCE GUIDE:
    Clear single pattern, no ambiguity         → 0.90–0.97
    Pattern present but some noise clips       → 0.80–0.89
    Ambiguous (needs elimination test)         → 0.50–0.65
    No pattern / insufficient clips            → 0.40–0.55
"""

In [56]:
prompt_v1 = f"""{CHIVA_RULES}
You are supposed to perform 2 tasks i.e. Shunt Classification and Appropriate Ligation 
=== ASSESSMENT: ===

═══════════════════════════════════════════════════════════════
STEP-BY-STEP DECISION GUIDE (Follow in order)
═══════════════════════════════════════════════════════════════

STEP 1: CHECK FOR EP N1→N2 (SFJ or Hunterian ENTRY)
    Look for: "EP N1→N2" with y≤0.098 (SFJ) or y≤0.353 (Hunterian)
    If YES with SFJ-ENTRY/Hunterian-ENTRY label → SFJ INCOMPETENT
    If NO  → SFJ COMPETENT (go to Case C)
    ✓ Found EP N1→N2? YES/NO

    STEP 2: IF YES to EP N1→N2, CHECK FOR REFLUX PATTERNS
    2a) ANY RP N3→N2 or RP N3→N1? (tributary reflux)
    2b) ANY RP N2→N1? (GSV reflux)
    2c) ANY RP anywhere else?
    2d) ANY EP N2→N3? (extra antegrade to tributary)

STEP 3: MATCH PATTERN TO TYPE

    ┌─ SFJ INCOMPETENT PATH (has EP N1→N2):
    │
    ├─ NO EP N2→N3:
    │  └─ Has RP N2→N1, no RP at N3 → TYPE 1 (confidence 0.90)
    │
    └─ YES EP N2→N3 EXISTS:
        ├─ Has RP N3 (at N2 or N1), NO RP N2→N1 → TYPE 3 (confidence 0.88)
        ├─ Has RP N3 AND RP N2→N1:
        │  ├─ eliminationTest absent → UNDETERMINED (confidence 0.55) [needs_elim_test=true]
        │  ├─ eliminationTest="Reflux" → TYPE 1+2 (confidence 0.80) 
        │  └─ eliminationTest="No Reflux" → TYPE 3 (confidence 0.75)

    ┌─ SFJ COMPETENT PATH (NO EP N1→N2):
    │
    ├─ EP N2→N3 EXISTS:
    │  └─ TYPE 2A (confidence 0.85-0.92)
    │     └─ Multiple RP at N3? → [ask_branching=true]
    │
    └─ ONLY EP N2→N2 (perforator entry):
        ├─ Has RP N3, NO RP N2→N1 → TYPE 2B (confidence 0.84)
        │  └─ Multiple RP at N3? → [ask_branching=true]
        ├─ Has RP N3 AND RP N2→N1 → TYPE 2C (confidence 0.82)
        │  └─ Multiple RP at N3? → [ask_branching=true]
        └─ No RP at all → NO SHUNT (confidence 0.95)

STEP 4: ASSIGN CONFIDENCE
    Clear pattern, no ambiguity → 0.90–0.97
    Pattern present, minor noise → 0.80–0.89
    Ambiguous / needs elimination test → 0.50–0.65
    Insufficient clips → 0.40–0.55

═══════════════════════════════════════════════════════════════
CRITICAL REMINDERS:
    • EP N1→N2 is THE KEY decision point — check this FIRST
    • EP N2→N2 means perforator (SFJ COMPETENT), never confuse with N1→N2
    • Type 2A has EP N2→N3; Type 2B/2C have EP N2→N2 (NOT N2→N3)
    • Type 2C differs from Type 1+2: 2C has EP N2→N2, Type 1+2 has EP N1→N2
    • Type 4/5 are N1→N3 path shunts and should be classified explicitly when present
    • RP only at N3 (not N2→N1) + EP N1→N2 = TYPE 3 (not 1+2)
═══════════════════════════════════════════════════════════════

=== TASK  ===
Follow the Step-by-Step Decision Guide above. 
Now remember the shunt type

=== LIGATION PLANNING FOR VENOUS SHUNT CLASSIFICATION ===

You are an expert vascular surgeon trained in CHIVA (hemodynamic conservative surgery) principles.
Your task is to generate a detailed, evidence-based ligation plan based on the shunt type and clinical findings.

=== SHUNT TYPE IDENTIFIED ===
Type: Shunt Type you classified in previous task


=== TASK ===
Based on the shunt type shunt_type, the clinical findings above, and the medical knowledge base provided:

1. Generate a detailed ligation plan with specific steps
2. Identify any additional clinical information needed
3. Consider complications and contraindications
4. Provide follow-up and monitoring recommendations
5. Consider CHIVA principles (hemodynamic, saphenous-vein-sparing when appropriate)

Important formatting rules:
1. ligation_steps must be a JSON array with one clear action per item.
2. Each ligation step must name the ligation point or vessel segment explicitly.
3. clinical_rationale must explain why that plan fits the shunt anatomy.
4. additional_info_needed must be [] when there is no meaningful extra information to request.
5. chiva_approach must describe the hemodynamic CHIVA reasoning, even if brief.

Output ONLY the Shunt Type and Ligation in the following format :
1. Type : ...
2. Ligation : ...
}}
}}"""

In [60]:
test_cases = {
    "Type 1": "Saphenofemoral junction reflux with GSV dilatation. Competent deep system. Direct backward flow from SFJ down the entire GSV with no tributary involvement. What is the shunt type and ligation strategy?",
    
    "Type 2A": "Saphenofemoral junction is competent. The greater saphenous vein has entry from the femoral vein at mid-thigh level through a large tributary. Reflux present in this tributary and GSV below entry point. Deep veins normal. What shunt type and ligation?",
}
other_cases = {
    "Type 2B": "SFJ is competent. A large perforating vein at Hunterian level connects femoral vein to GSV with significant reflux. GSV distal to entry shows retrograde flow into tributaries. No deep system involvement. Classify and recommend ligation.",
    
    "Type 2C": "SFJ competent. A medial thigh perforator feeds the GSV at mid-calf level with reflux propagating both proximally and distally. Associated tributary incompetence. What is this shunt type and management?",
    
    "Type 1+2": "SFJ incompetent with proximal GSV reflux. Additionally, mid-thigh perforator feeds GSV with secondary reflux into tributaries distally. Dual entry points. Classify and suggest complete ligation strategy.",
    
    "Type 3": "Incompetent calf perforators between gastrocnemius and soleus muscles communicating directly with deep posterior tibial veins. Bidirectional flow with reflux during calf compression. No superficial entry points. What type and approach?",
    
    "Type 4": "Pelvic venous insufficiency with ovarian/gonadal vein reflux crossing midline and draining via broad ligament veins into superficial system. SFJ competent. Lower abdominal symptoms with leg symptoms. Type and management?",
    
    "Type 5": "Laparoscopic findings: incompetent retroperitoneal veins draining into pelvic vasculature with secondary superficial reflux. Patent foramen ovale anatomy involved. Type classification?",
    
    "Type 6": "Direct reentry point: GSV incompetence with ectopic branch entering lateral femoral vein below SFJ. Reflux pattern unusual with entry below anatomic SFJ. Type and ligation site?",

    "No shunt": "Patient with normal duplex findings. Saphenofemoral junction is fully competent with no reflux. Greater saphenous vein shows normal antegrade flow during compression. No perforator incompetence. Deep veins patent with normal flow dynamics. All valves functioning properly. What is the diagnosis and recommended management?"
}


def answer_case(case_description):
    prompt = prompt_v1
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        #max_length=300,
        temperature=0.8,
        top_p=1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Run tests
for shunt_type, case in test_cases.items():
    print(f'\n{"="*80}')
    print(f'SHUNT TYPE: {shunt_type}')
    print(f'{"="*80}')
    print(f'CASE:\n{case}\n')
    print(f'MODEL ANSWER:')
    response = answer_case(case)
    print(response)
    print()


SHUNT TYPE: Type 1
CASE:
Saphenofemoral junction reflux with GSV dilatation. Competent deep system. Direct backward flow from SFJ down the entire GSV with no tributary involvement. What is the shunt type and ligation strategy?

MODEL ANSWER:

=== CHIVA VENOUS SHUNT CLASSIFICATION RULES ===

ANATOMY:
    N1 = Deep venous system (femoral / popliteal vein)
    N2 = Great Saphenous Vein (GSV) or Small Saphenous Vein (SSV) trunk
    N3 = Tributaries / superficial branches
    EP = Physiological (forward, antegrade) flow — NORMAL clip
    RP = Retrograde (pathological, reflux) flow — ABNORMAL clip
    SFJ = Saphenofemoral Junction  →  posYRatio ≤ 0.098
    Hunterian Perforator            →  0.098 < posYRatio ≤ 0.353

═══════════════════════════════════════════════════════════
CRITICAL RULE — SFJ COMPETENCE (read before classifying):
    SFJ is INCOMPETENT if and only if a clip has fromType=N1 AND toType=N2 (EP N1→N2).
    EP N2→N2 means blood circulates within the saphenous trunk via a perf

# Script 2 — Fine-Tuning Mistral-7B-Instruct · Venous Medicine
Retraining to make the model more specific to shunt classification and ligation

In [73]:
"""
PASTE THIS INTO A JUPYTER CELL IN YOUR NOTEBOOK
This trains the already fine-tuned model on reasoning tasks
"""

import sys
import os

# Add Windows path (accessible via WSL2 mount) to Python path
wsl_path = "/mnt/c/Users/Krish/Downloads/LLM_Finetuning"
sys.path.insert(0, wsl_path)
os.chdir(wsl_path)

print(f"Working directory: {os.getcwd()}")
print(f"Files in directory: {[f for f in os.listdir('.') if f.startswith('training_')]}")

# ============================================================================
# STEP 1: IMPORT AND LOAD DATA
# ============================================================================

from training_data_comprehensive import generate_comprehensive_training_pairs
from datasets import Dataset
import numpy as np

# Generate training data
training_pairs = generate_comprehensive_training_pairs()
print(f"Generated {len(training_pairs)} training pairs")

# Split into train/eval
np.random.seed(42)
indices = np.random.permutation(len(training_pairs))
train_size = int(0.8 * len(training_pairs))
train_indices = indices[:train_size]
eval_indices = indices[train_size:]

train_data = [training_pairs[i] for i in train_indices]
eval_data = [training_pairs[i] for i in eval_indices]

train_dataset = Dataset.from_dict({
    "text": [ex["text"] for ex in train_data],
})

eval_dataset = Dataset.from_dict({
    "text": [ex["text"] for ex in eval_data],
})

print(f"Training pairs: {len(train_dataset)}")
print(f"Eval pairs: {len(eval_dataset)}")

# ============================================================================
# STEP 2: SETUP TRAINING (assumes model, tokenizer already loaded)
# ============================================================================

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from pathlib import Path

output_dir = "./reasoning_finetuned_output"
Path(output_dir).mkdir(exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=5,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=1,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=False,
    max_grad_norm=1.0,
    seed=42,
    bf16=True,
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
)

# ============================================================================
# STEP 3: TOKENIZE DATA AND CREATE TRAINER
# ============================================================================

# Note: tokenizer should already be loaded in your notebook from the cell where you loaded the model

# Tokenize the datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
eval_dataset_tokenized = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    data_collator=data_collator,
)

print("Starting training...")
train_result = trainer.train()

print(f"\nTraining completed!")
print(f"Final training loss: {train_result.training_loss:.4f}")

# ============================================================================
# STEP 4: SAVE REFINED MODEL
# ============================================================================

final_output_dir = "./mistral_reasoning_enhanced"
Path(final_output_dir).mkdir(exist_ok=True)

model.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

print(f"✓ Model saved to {final_output_dir}")

# ============================================================================
# STEP 5: QUICK TEST ON NEW CASE
# ============================================================================

print("\n" + "="*80)
print("QUICK INFERENCE TEST")
print("="*80)

test_prompt = """[INST] === SHUNT CLASSIFICATION ===
Clips:
  Clip 00: EP N1→N2  y=0.080 [SFJ-ENTRY=INCOMPETENT]
  Clip 01: RP N2→N1  y=0.300 [GSV-TRUNK-REFLUX: N2→N1]

Classify using CHIVA rules. Provide type, confidence, and reasoning. [/INST]"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.3, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "[/INST]" in response:
    response = response.split("[/INST]")[1].strip()

print("Test Response:")
print(response)

Working directory: /mnt/c/Users/Krish/Downloads/LLM_Finetuning
Files in directory: ['training_cell.py', 'training_data_comprehensive.py', 'training_data_reasoning.py']
Generated 17 training pairs
Training pairs: 13
Eval pairs: 4


Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 31.84 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 44.86 GiB is allocated by PyTorch, and 287.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import shutil
import os

src = '/home/krish/finetuning/venv_pytorch/Include'
dst = '/mnt/c/Users/Krish/Downloads/LLM_Finetuning/Include_backup'

shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"✓ Copied entire Include folder to Windows")
print(f"Location: C:\\Users\\Krish\\Downloads\\LLM_Finetuning\\Include_backup")


# Training Everything from beginning to suit our task
Retraining to make the model more specific to shunt classification and ligation

In [2]:
! python3 -m pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp313-cp313-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/torchvision-0.2.0-py2.py3-none-any.whl (48 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 8.4 MB/s  0:00:02 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 11.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 9.9 MB/s  0:00:01 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 9.3 MB/

In [7]:
! python3 -m pip install transformers peft datasets

  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached regex-2026.4.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached torch-2.11.0-cp313-cp313-manylinux_2_

In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.current_device()}")
print(f"Device name: {torch.cuda.get_device_name()}")

CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 5090


/home/krish/miniconda/lib/python3.13/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [4]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "transformers", "peft", "datasets"])
print("Done")

# Put all data and other files in the correct folders and run these cells
import torch, json
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType

TRAIN_FILE = "./training_data.jsonl"
VAL_FILE = "./validation_data.jsonl"

def load_jsonl(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

train_pairs = load_jsonl(TRAIN_FILE)
val_pairs = load_jsonl(VAL_FILE)

print(f"Loaded {len(train_pairs)} training, {len(val_pairs)} validation")

Done


/home/krish/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: './training_data.jsonl'

In [27]:
def format_instruction_response(pair):
    instruction = pair.get("instruction", "").strip()
    input_text = pair.get("input", "").strip()
    output = pair.get("output", "").strip()
    if input_text:
        prompt = f"[INST] {instruction}\n\n{input_text} [/INST]"
    else:
        prompt = f"[INST] {instruction} [/INST]"
    return f"{prompt} {output}"

train_texts = [format_instruction_response(p) for p in train_pairs]
val_texts = [format_instruction_response(p) for p in val_pairs]
print(f"Formatted {len(train_texts)} training, {len(val_texts)} validation")

Formatted 841 training, 361 validation


In [ ]:
MODEL_PATH = "./Models"

print(f"Loading local model from {MODEL_PATH}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,
    device_map=None,
)
model = model.to("cuda:0")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

print("Local model loaded successfully")

Loading local model from ./Models...


Loading weights:  58%|█████████████████████████████████████████████▉                                 | 169/291 [00:05<00:04, 25.73it/s]